# 02 - Pre-quantize LLM backbones to 4-bit AWQ

Populates `/MyDrive/quest_kg/models/` with quantized weights for:
- LLaMA-3.2-3B-Instruct (~2 GB)
- Mistral-7B-Instruct-v0.3 (~4 GB)
- Qwen-2.5-7B-Instruct (~4 GB)
- LLaMA-3.1-8B-Instruct (~5 GB)

Total ~15 GB Drive footprint. After this runs once, every experiment notebook
loads these directly from Drive — no re-downloading per session.

**Prerequisites:**
- Run `00_setup.ipynb` first (mounts Drive, installs deps, logs into HF).
- HF token must have access to gated LLaMA-3.1 and LLaMA-3.2 models.
- GPU runtime (T4 minimum; A100 strongly preferred for quantization speed).

In [ ]:
import os
os.chdir('/content/quest-kg-cikm2026')
MODELS_ROOT = '/content/drive/MyDrive/quest_kg/models'
print('writing to:', MODELS_ROOT)

## Confirm GPU

In [ ]:
from quest_kg.utils.device import cuda_info
cuda_info()

## Quantize all four backbones

If a pre-quantized AWQ mirror exists on HF, the script will download it (fast, ~5 min each).
Otherwise it quantizes locally with AutoAWQ (~10-60 min each depending on GPU).

In [ ]:
!python -m scripts.quantize.quantize_awq \
    --root $MODELS_ROOT \
    --models llama32-3b mistral-7b qwen25-7b llama31-8b

## Smoke-load each model to confirm they work

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, os

for shortname in ['llama32-3b', 'mistral-7b', 'qwen25-7b', 'llama31-8b']:
    path = f'{MODELS_ROOT}/{shortname}'
    if not os.path.exists(f'{path}/.done'):
        print(f'MISS {shortname}: not quantized')
        continue
    try:
        tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            path, device_map='cuda', trust_remote_code=True,
        )
        prompt = 'The capital of France is'
        ids = tok(prompt, return_tensors='pt').to('cuda')
        out = model.generate(**ids, max_new_tokens=10, do_sample=False)
        print(f'OK  {shortname}: {tok.decode(out[0], skip_special_tokens=True)!r}')
        del model, tok
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'FAIL {shortname}: {e}')